<h1 align="center" style="font-size:50px;">
<b>LLMOps</b>
</h1>

En este notebook, exploraremos como podemos crear un pipeline de LLM Operations para automatizar el entrenamiento, deployment y versionado de artefactos durante el ciclo de vida de un LLM.

Utilizaremos Kubeflow y Vertex Pipeines para crear un worfklow que resuelva las tareas mencionadas. Usaremos tambien herramientas del stack de VertexAI, cubiertas previamente, para versionar prompts y modelos durante el pipeline.

En este caso, imaginaremos que estamos trabajando con un asistente de código, especializaco para respuesta de preguntas relacionadas con programación. Asumimos tambien que el mismo fue finetuneado con el dataset público de StackOverflow para mejorar sus respuestas.

A continuación desarrollaremos un posible pipeline de entrenamiento y deployment de este asistente.



## **0) Carga de datos para el finetuning**

In [40]:
import pandas as pd
import random
import time

from google import genai
from google.cloud import aiplatform, bigquery
import numpy as np
import tqdm

In [41]:
bq_client = bigquery.Client(project="notifications-me")

In [42]:
# GCP configs

# get project ID
PROJECT_ID = ! gcloud config get project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"
if PROJECT_ID == "(unset)":
    print(f"Please set the project ID manually below")

# define project information
if PROJECT_ID == "(unset)":
    PROJECT_ID = "notifications-me"  # @param {type:"string"}

# generate an unique id for this session
from datetime import datetime

UID = datetime.now().strftime("%m%d%H%M")

Please set the project ID manually below


In [43]:
def load_training_data(questions_size=100):
  QUERY_TEMPLATE = """
          SELECT distinct q.id, q.title, q.body, q.accepted_answer_id
          FROM (SELECT * FROM `bigquery-public-data.stackoverflow.posts_questions`
          where Score > 0 ORDER BY View_Count desc) AS q
          LIMIT {limit} ;
          """
  query = QUERY_TEMPLATE.format(limit=questions_size)
  query_job = bq_client.query(query)
  rows = query_job.result()

  return rows.to_dataframe()

In [44]:
df = load_training_data()
df

Forbidden: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/notifications-me/jobs?prettyPrint=false: Access Denied: Project notifications-me: User does not have bigquery.jobs.create permission in project notifications-me.

Location: None
Job ID: 1169ab93-8008-4679-b49d-91d7b745f4b9


## **1) Finetuning**



Por motivos de costos y tiempos, no vamos a estar finetuneando un modelo en esta oportunidad. El objetivo de esta seccion es ilustrar como con una funcion placeholder, podriamos ejecutar el finetuning de nuestro modelo dentro del workflow del pipeline

In [ ]:
def get_model_and_tokenizer():
  # cargamos el modelo y el tokenizer del hub
  print("Getting model from the hub / ai studio")

def get_lora_peft_():
  # creamos un PEFT model con LoRA
  print("Creating LoRA PEFT representation")


In [ ]:
def fine_tune_big_model():
  # esta funcion representa los pasos a ejecutar durante el finetuning
  get_model_and_tokenizer()
  get_lora_peft_()

  print("Finetuning big model")
  print("Finetuning done! ✅")

## **2) Versionado de prompts**


In [ ]:
def save_training_prompts():
  from vertexai.preview import prompts
  from vertexai.preview.prompts import Prompt

  user_message = "user_message"
  best_q_title = "best_q_title"
  best_q_body = "best_q_body"
  best_a_response = "best_a_response"

  system_prompt = 'You are an expert AI chatbot that is here to help \n      with stackoverflow QA. You will be prompted to help with a specific topic\n      and receive information from a database with existing stackoverflow QAs.\n      You must use that information to help the user, but NOT necessarily retrieve\n      the exact information you get from the database, use it for grounding.\n\n      You CAN answer several different questions in a same chat context, but always\n      to coding/tech/stackoverflow related subjects.\n '

  prompt = f'''
      You are an expert AI chatbot that is here to help with stackoverflow QA.

      Here's the user question: {user_message}.

      This is the most similar question title found in the vector database: {best_q_title}.

      This is the most similar question's body: {best_q_body}

      This is the best answer to that specific question, taken from the stackoverflow QA database: {best_a_response}

      ONLY IF RELEVANT INFORMATION WAS RETRIEVED FROM THE QA DATABSE, use this information to answer the user's query, else proceed with answer. please
      remember that the answer provided from database not necesarily matches
      the user context PERFECTLY (i.e: maybe the user is using different variabels, connecting to different databases, etc) you must use
      the anwer information to get a better understanding of the problem, but you should rephrase it using your best knowledge to give the
      user the best response.
    '''

  # create a prompt object for our model
  cloud_prompt = Prompt(
      prompt_name="QA-demo-prompt",
      prompt_data=prompt,
      model_name='gemini-2.0-flash-001',
      system_instruction=system_prompt, # system prompt
  )


  # Save Prompt to online resource.
  saved_cloud_prompt = prompts.create_version(prompt=cloud_prompt)

  print("\nSaved prompt into Prompt Manager ✅")

In [ ]:
save_training_prompts()

## **3) Construyendo los componentes del pipeline**

En este ejemplo, utilizaremos kubeflow pipelines para integrar nuestro workflow directamente en VertexAI.

Para esto, primero debemos crear _componentes_ que luego iremos ejecutando de manera secuencial (tambien se puede en paralelo) en nuestro pipeline.

In [ ]:
!pip install kfp

In [ ]:
from kfp import dsl
from kfp import compiler

# Ignore FutureWarnings in kfp
import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module='kfp.*'
)

In [ ]:
@dsl.component
def get_data_component()-> bool:
  train_df = load_training_data()
  return True

In [ ]:
@dsl.component
def version_training_prompts(is_data_loading_ready: bool) -> bool:
  save_training_prompts()
  return True

In [ ]:
@dsl.component
def simulated_finetuning_component(is_versioning_ready: bool):
  fine_tune_big_model()


Luego de usar el decorator de componentes en las funciones, las mismas dejan de devolver su "output" y en cambio devuelven un _pipeline_task_ object.

## **4) Construyendo el pipeline**

Primero **declaramos** la logica del pipeline utilizando el decorator dsl.pipeline. esto actuara como un "blueprint" para el pipeline que vamos a generar.

En caso de tener errores en las definiciones o los inputs/outputs de nuestros componentes, en este paso nos deberian aparecer errores.

In [ ]:
@dsl.pipeline
def finetuning_pipeline():
  get_data_task = get_data_component()

  if get_data_task.output:
    versioning_task = version_training_prompts(is_data_loading_ready=get_data_task.output)
    simulated_finetuning_component(is_versioning_ready=versioning_task.output)

  print("Pipeline declared succesfully ✅")

Una vez que tenemos el pipeline (blueprint) declarado, "compilamos" el mismo a un archivo yaml para su posterior deployment

In [ ]:
compiler.Compiler().compile(finetuning_pipeline, 'finetuning_pipeline.yaml')
#!cat finetuning_pipeline.yaml

## **6) Deployeando el pipeline de entrenamiento en Vertex**

Ahora que ya tenemos el pipeline "compilado" estamos listos para hacer un deployment del mismo directo al engine de vertex.

In [ ]:
#!gcloud services enable compute.googleapis.com aiplatform.googleapis.com storage.googleapis.com bigquery.googleapis.com --project {PROJECT_ID}

In [ ]:
from google.cloud.aiplatform import PipelineJob

job = PipelineJob(
        template_path="finetuning_pipeline.yaml",
        display_name=f"test_finetuning_pipeline",
        #parameter_values=pipeline_arguments, en caso de que nuestro pipeline requiera params. Se declaran como args de la funcion decorada
        location="us-central1",
        #pipeline_root="./", donde guardar artefactos intermedios, nosotros no tenemos pero puede ser un GCS bucket
)

In [ ]:
job.submit()
job.state

<img src='https://storage.googleapis.com/humai-datasets/imagenes/llms/6_LLMOps/0_image.png' />

## **7) Simulando Monitoring y observability con LangSmith**

Utilizaremos LangSmith, una herramienta de prompt management, monitoring y observability para realizar un control sobre la calidad y evolución de las respuestas de nuestro deployment

In [ ]:
!pip install -U langchain langchain-openai

In [ ]:
%env LANGSMITH_TRACING=true

In [ ]:
%env LANGSMITH_ENDPOINT="https://api.smith.langchain.com"

In [ ]:
%env LANGSMITH_API_KEY="..."

In [ ]:
%env LANGSMITH_PROJECT="llmtest"

In [ ]:
%env OPENAI_API_KEY="..."

LangSmith nos permite agregar **"traces"** para observabilidad de los inputs y outputs de nuestros modelos deployeados. Tiene una integración nativa con langchain, pero tambien podemos usar el decorator "traceable" para hacer lo mismo:

In [ ]:
OPENAI_API_KEY="..."

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(api_key=OPENAI_API_KEY)
llm.invoke("Esto es un test, estoy probando langsmith con langchain")

In [ ]:
import openai
from langsmith.wrappers import wrap_openai
from langsmith import traceable

client = wrap_openai(openai.Client(api_key=OPENAI_API_KEY))

@traceable(run_type="tool")
def call_tool():
  return "simple tool response"

@traceable # Auto-trace this function
def testing_smith(user_input: str):
    response = call_tool()
    result = client.chat.completions.create(
        messages=[{"role": "user", "content": user_input + response}],
        model="gpt-3.5-turbo"
    )
    return result.choices[0].message.content

testing_smith("Hola soy lucas vos quien sos")
# Out:  Hello there! How can I assist you today?

## **Cleaning up**

In [ ]:
# delete Cloud Storage bucket
! gsutil rm -r {BUCKET_URI}